# 06b Diagnostics: why is INT8 more accurate than FP32 on z and r?

Checks: signed bias per parameter, train vs test error (overfitting), weight only vs activation only quantisation, and the z and r error correlation.

In [1]:
%run ./00_common.ipynb

C:\Users\Rey\anaconda3\envs\kolb\Lib\site-packages\nbformat\validator.py:434: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  _validate(nbdict, ref, version, version_minor, relax_add_props)


PROJECT  : C:\Users\Rey\Documents\Kolb_Radar_Project
FAKE_MODE: False
[mitsuba] Warning: Couldn't import the ipywidgets package. Installing this package is required for the system to properly log messages and print in Jupyter notebooks!


torch device: cuda
mitsuba variant: cuda_ad_rgb


Switched off Tx:  2
Switched off Rx:  10
torch 2.5.1+cu121 | device cuda
common ready


In [2]:
import copy
nbg = json.load(open(PROJECT / 'notebooks' / '06_quantise_8bit.ipynb', encoding='utf-8'))
src = '\n'.join(''.join(c['source']) for c in nbg['cells'] if c['cell_type'] == 'code' and 'def fold_bn' in ''.join(c['source']))
src = src.split("folded_check =")[0]          # only the function and class definitions
X, P = load_dataset(); tr, va, te = split_indices(len(X))
exec(src)
fp32 = OneShotNet().to(DEVICE); fp32.load_state_dict(torch.load(MODELS / 'oneshot_fp32.pt', map_location=DEVICE)); fp32.eval()
def predict(m, idx, bs=256):
    out = []
    with torch.no_grad():
        for i in range(0, len(idx), bs):
            out.append(m(torch.from_numpy(X[idx[i:i+bs]].astype(np.float32)).to(DEVICE)).cpu().numpy())
    return from_unit(np.concatenate(out))
def stats(name, m):
    rows = {}
    for split, idx in (('train', tr[:1000]), ('test', te)):
        p = predict(m, idx); e = (p - P[idx]) * 1000
        rows[split] = {'mae': dict(zip(PARAM_NAMES, np.abs(e).mean(0).round(2).tolist())),
                       'bias': dict(zip(PARAM_NAMES, e.mean(0).round(2).tolist())),
                       'corr_z_r_err': float(np.corrcoef(e[:, 2], e[:, 3])[0, 1]),
                       'front_surface_mae_mm': float(np.abs(e[:, 2] - 2 * e[:, 3]).mean()),
                       'front_surface_baseline_mae_mm': float(np.abs((P[idx, 2] - 2 * P[idx, 3]) - (P[tr, 2] - 2 * P[tr, 3]).mean()).mean() * 1000)}
    print(name, json.dumps(rows)); return rows
out = {'fp32': stats('FP32', fp32)}
q8 = build_ptq(fp32, 8); out['int8'] = stats('INT8 w+a', q8)
# weight only: disable activation quantisation
qw = build_ptq(fp32, 8)
for mm in qw.modules():
    if isinstance(mm, QAct): mm.calib = True      # calib mode passes activations through unchanged
out['int8_weights_only'] = stats('INT8 weights only', qw)
# activation only: weights at 16 bit (almost exact)
qa = build_ptq(fp32, 16); out['int8_acts_only'] = stats('INT8 activations only', qa)
# output saturation check
p = predict(fp32, te); u = to_unit(p)
out['fp32_test_unit_outputs_near_0_or_1'] = float(((u < 0.02) | (u > 0.98)).mean())
print('fraction of saturated sigmoid outputs', out['fp32_test_unit_outputs_near_0_or_1'])
save_json(out, '06b_diagnostics.json')

FP32 {"train": {"mae": {"x": 2.7799999713897705, "y": 10.119999885559082, "z": 5.090000152587891, "r": 2.490000009536743}, "bias": {"x": 1.0299999713897705, "y": -0.6700000166893005, "z": -2.4100000858306885, "r": -0.6800000071525574}, "corr_z_r_err": 0.9840321086988758, "front_surface_mae_mm": 1.2333574295043945, "front_surface_baseline_mae_mm": 11.528750881552696}, "test": {"mae": {"x": 2.799999952316284, "y": 10.1899995803833, "z": 5.110000133514404, "r": 2.5799999237060547}, "bias": {"x": 1.100000023841858, "y": 0.3700000047683716, "z": -1.7899999618530273, "r": -0.4000000059604645}, "corr_z_r_err": 0.9856333677649433, "front_surface_mae_mm": 1.222639560699463, "front_surface_baseline_mae_mm": 12.11738120764494}}
INT8 w+a {"train": {"mae": {"x": 2.940000057220459, "y": 10.119999885559082, "z": 5.159999847412109, "r": 2.5199999809265137}, "bias": {"x": 1.0700000524520874, "y": -0.699999988079071, "z": -2.630000114440918, "r": -0.8299999833106995}, "corr_z_r_err": 0.9846208367972971,

INT8 weights only {"train": {"mae": {"x": 2.759999990463257, "y": 10.119999885559082, "z": 5.110000133514404, "r": 2.490000009536743}, "bias": {"x": 0.9900000095367432, "y": -0.6499999761581421, "z": -2.450000047683716, "r": -0.699999988079071}, "corr_z_r_err": 0.9839098696477, "front_surface_mae_mm": 1.2485769987106323, "front_surface_baseline_mae_mm": 11.528750881552696}, "test": {"mae": {"x": 2.7799999713897705, "y": 10.1899995803833, "z": 5.119999885559082, "r": 2.5799999237060547}, "bias": {"x": 1.0700000524520874, "y": 0.38999998569488525, "z": -1.840000033378601, "r": -0.4099999964237213}, "corr_z_r_err": 0.9855212309746899, "front_surface_mae_mm": 1.2360581159591675, "front_surface_baseline_mae_mm": 12.11738120764494}}
INT8 activations only {"train": {"mae": {"x": 2.9200000762939453, "y": 10.119999885559082, "z": 5.150000095367432, "r": 2.509999990463257}, "bias": {"x": 1.059999942779541, "y": -0.7200000286102295, "z": -2.5799999237060547, "r": -0.8199999928474426}, "corr_z_r_e